# Phase 6 · Notebook 01 — LegalBERT Fine-Tune

In Phase 2 we fine-tuned `roberta-base` on TAB train. `roberta-base` was
pre-trained on web crawl + books — general-purpose English. A reasonable
question: would a backbone *pre-trained on legal text* do better?

[`nlpaueb/legal-bert-base-uncased`](https://huggingface.co/nlpaueb/legal-bert-base-uncased)
is BERT-base pre-trained on 12 GB of legal corpora (US court cases, EU
legislation, US contracts). Same architecture class as `roberta-base`,
different training corpus. Drop it into the Phase 2 recipe and see what
happens.

Expected lift: 1–3 F1 on legal-specific entities (PERSON, ORG, MISC).
Marginal effect on domain-agnostic categories (DATETIME, QUANTITY).

---


In [1]:
# ── Run me first if you're on Colab (skip locally — already in requirements.txt) ──
# !pip -q install transformers datasets evaluate seqeval accelerate spacy
# !python -m spacy download en_core_web_lg


## Setup


In [2]:
import sys
sys.path.insert(0, "../../src")

import time
import numpy as np
import pandas as pd
import torch
import warnings
warnings.filterwarnings("ignore")

from anonymisation.data import load_tab
from anonymisation.iob import (
    BIO_LABELS, LABEL_TO_ID, ID_TO_LABEL, TAB_ENTITY_TYPES,
    gold_spans_for_training, offsets_to_bio,
)
from anonymisation.evaluation import evaluate_document, merge_results, results_to_dataframe
from anonymisation.predictors import make_finetuned_predictor
from anonymisation.device import best_device, report_device

print(report_device())


Device: mps    (Apple Silicon (MPS))


## Configuration

The only meaningful difference from Phase 2 notebook 03 is `BASE_MODEL`.
Same epoch count, same learning rate, same batch size, same eval recipe —
so the comparison is fair.


In [3]:
BASE_MODEL  = "nlpaueb/legal-bert-base-uncased"
MAX_LENGTH  = 384
STRIDE      = 64
NUM_EPOCHS  = 3
LR          = 2e-5
BATCH_SIZE  = 16
SEED        = 42

OUTPUT_DIR   = "checkpoints/legalbert-tab"
RESULTS_PATH = "../results/legalbert_results.csv"

device, _ = best_device()
if device != "cuda":
    BATCH_SIZE = 8
    print(f"Non-CUDA device — reducing batch size to {BATCH_SIZE}")


Non-CUDA device — reducing batch size to 8


## Load TAB + tokenise


In [4]:
dataset = load_tab()
print({split: len(dataset[split]) for split in dataset})


{'train': 1112, 'validation': 541, 'test': 555}


In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenise_and_align(doc):
    text = doc["text"]
    spans = gold_spans_for_training(doc)
    enc = tokenizer(
        text,
        return_offsets_mapping=True,
        return_overflowing_tokens=True,
        truncation=True,
        max_length=MAX_LENGTH,
        stride=STRIDE,
        padding=False,
    )
    examples = []
    for chunk_idx in range(len(enc["input_ids"])):
        offsets = enc["offset_mapping"][chunk_idx]
        word_ids = enc.word_ids(batch_index=chunk_idx)
        labels = offsets_to_bio(offsets, spans, word_ids=word_ids)
        examples.append({
            "input_ids":      enc["input_ids"][chunk_idx],
            "attention_mask": enc["attention_mask"][chunk_idx],
            "labels":         labels,
        })
    return examples


def build_split(split_name, max_docs=None):
    out = []
    docs = list(dataset[split_name])
    if max_docs:
        docs = docs[:max_docs]
    for doc in docs:
        out.extend(tokenise_and_align(doc))
    return out


print("Tokenising train + validation (~1 min)...")
train_examples = build_split("train")
val_examples   = build_split("validation")
print(f"  train chunks: {len(train_examples):,}")
print(f"  val   chunks: {len(val_examples):,}")


Tokenising train + validation (~1 min)...
  train chunks: 6,069
  val   chunks: 1,851


In [6]:
from datasets import Dataset
train_ds = Dataset.from_list(train_examples)
val_ds   = Dataset.from_list(val_examples)


## Build the model


In [7]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(BIO_LABELS),
    id2label=ID_TO_LABEL,
    label2id=LABEL_TO_ID,
)
print(f"Model: {BASE_MODEL}, num_labels={len(BIO_LABELS)}")


Some weights of BertForTokenClassification were not initialized from the model checkpoint at nlpaueb/legal-bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model: nlpaueb/legal-bert-base-uncased, num_labels=17


## Train


In [8]:
from transformers import (
    DataCollatorForTokenClassification, TrainingArguments, Trainer
)
import evaluate as hf_evaluate

collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
seqeval = hf_evaluate.load("seqeval")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    true_labels, true_preds = [], []
    for p_seq, l_seq in zip(preds, labels):
        tl, tp = [], []
        for p, l in zip(p_seq, l_seq):
            if l == -100:
                continue
            tl.append(ID_TO_LABEL[int(l)])
            tp.append(ID_TO_LABEL[int(p)])
        true_labels.append(tl)
        true_preds.append(tp)
    res = seqeval.compute(predictions=true_preds, references=true_labels)
    return {
        "f1":        res["overall_f1"],
        "precision": res["overall_precision"],
        "recall":    res["overall_recall"],
    }


args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    seed=SEED,
    report_to="none",
    fp16=(device == "cuda"),
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)
print("Starting training...")
trainer.train()
print("\nDone. Best checkpoint loaded.")


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting training...


Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,0.087800,0.109382,0.802305,0.789622,0.815401
2,0.089100,0.108221,0.811774,0.809175,0.814390
3,0.064200,0.112476,0.801241,0.802970,0.799520



Done. Best checkpoint loaded.


## Save the model — needed by Notebook 02 (ensemble) downstream


In [9]:
trainer.save_model(OUTPUT_DIR + "/final")
tokenizer.save_pretrained(OUTPUT_DIR + "/final")
print(f"Saved → {OUTPUT_DIR}/final")


Saved → checkpoints/legalbert-tab/final


## Evaluate on TAB test


In [10]:
predict = make_finetuned_predictor(model, tokenizer, device=device, max_length=MAX_LENGTH, stride=STRIDE)

test_docs = list(dataset["test"])
print(f"Evaluating on {len(test_docs)} test documents...")

all_merged = {}
for mode in ["partial", "exact"]:
    print(f"\n--- {mode} match ---")
    per_doc = []
    start = time.time()
    for i, doc in enumerate(test_docs):
        if (i + 1) % 50 == 0:
            elapsed = time.time() - start
            print(f"  {i + 1}/{len(test_docs)}  ({(i + 1)/elapsed:.1f} docs/s)")
        per_doc.append(evaluate_document(predict, doc, mode=mode))
    print(f"  done in {time.time() - start:.1f}s")
    all_merged[mode] = merge_results(per_doc)

results_df = results_to_dataframe(all_merged)
results_df.insert(0, "model", "legalbert_finetuned_tab")
results_df.to_csv(RESULTS_PATH, index=False)
print(f"\nSaved → {RESULTS_PATH}")


Evaluating on 555 test documents...

--- partial match ---
  50/555  (9.2 docs/s)
  100/555  (8.9 docs/s)
  150/555  (8.7 docs/s)
  200/555  (8.9 docs/s)
  250/555  (9.5 docs/s)
  300/555  (10.1 docs/s)
  350/555  (10.5 docs/s)
  400/555  (11.3 docs/s)
  450/555  (12.0 docs/s)
  500/555  (11.2 docs/s)
  550/555  (10.9 docs/s)
  done in 51.0s

--- exact match ---
  50/555  (9.2 docs/s)
  100/555  (8.9 docs/s)
  150/555  (8.7 docs/s)
  200/555  (8.9 docs/s)
  250/555  (9.5 docs/s)
  300/555  (10.2 docs/s)
  350/555  (10.6 docs/s)
  400/555  (11.4 docs/s)
  450/555  (12.1 docs/s)
  500/555  (11.4 docs/s)
  550/555  (11.1 docs/s)
  done in 50.1s

Saved → ../results/legalbert_results.csv


## Per-entity results


In [11]:
from anonymisation.mapping import TAB_TO_SPACY

for mode in ["partial", "exact"]:
    merged = all_merged[mode]
    print(f"\n── {mode.upper()} MATCH ──")
    rows = []
    for et in list(TAB_TO_SPACY.keys()) + ["_ALL"]:
        r = merged[et]
        rows.append({
            "Entity": et if et != "_ALL" else "▶ OVERALL",
            "TP": r.tp, "FP": r.fp, "FN": r.fn,
            "Precision": f"{r.precision:.1%}",
            "Recall":    f"{r.recall:.1%}",
            "F1":        f"{r.f1:.1%}",
        })
    print(pd.DataFrame(rows).to_string(index=False))



── PARTIAL MATCH ──
   Entity    TP   FP   FN Precision Recall    F1
   PERSON  3847  593  291     86.6%  93.0% 89.7%
      ORG  1191  699  755     63.0%  61.2% 62.1%
      LOC  1152  484  307     70.4%  79.0% 74.4%
 DATETIME  9282  993  254     90.3%  97.3% 93.7%
 QUANTITY   536  242  128     68.9%  80.7% 74.3%
     CODE  1571  115   41     93.2%  97.5% 95.3%
      DEM   352  284  565     55.3%  38.4% 45.3%
     MISC    30  153  507     16.4%   5.6%  8.3%
▶ OVERALL 17961 3563 2848     83.4%  86.3% 84.9%

── EXACT MATCH ──
   Entity    TP   FP   FN Precision Recall    F1
   PERSON  3614  826  524     81.4%  87.3% 84.3%
      ORG   932  958 1014     49.3%  47.9% 48.6%
      LOC  1035  601  424     63.3%  70.9% 66.9%
 DATETIME  8893 1382  643     86.5%  93.3% 89.8%
 QUANTITY   411  367  253     52.8%  61.9% 57.0%
     CODE  1518  168   94     90.0%  94.2% 92.1%
      DEM   203  433  714     31.9%  22.1% 26.1%
     MISC     4  179  533      2.2%   0.7%  1.1%
▶ OVERALL 16610 4914 4199    

## What to look for

- **Overall F1 vs the Phase 2 RoBERTa run.** Compare `_ALL` partial-match F1 directly with `../../phase2_baseline_comparison/results/finetuned_results.csv`. LegalBERT is reported to lift legal-NER F1 by 1–3 points; if you see less, the domain match isn't strong enough to justify a different backbone for this corpus.

- **Per-entity-type lift.** PERSON, ORG, MISC are where LegalBERT's pretraining advantage should show. DATETIME, QUANTITY, CODE: probably no meaningful change.

- **Boundary errors (exact-match vs partial).** If LegalBERT closes the partial/exact gap, that's a real win. If the gap stays similar, a CRF head (Phase 6.1) is the next intervention.

Notebook 02 (ensemble) reads this CSV — run that next.
